In [32]:
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [26]:
base_url = "http://localhost:11434"
model = "llama3.2"

llm = ChatOllama(base_url=base_url,
                 model = model,
                 temperature=0)

parser = StrOutputParser()




In [42]:

class FeedBack(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description="Give the sentiment of the feedback")


prompt1 = PromptTemplate(template="Classify the sentiment (positive or negative):\n{text}", 
                         input_variables=["text"])

classifier_chain = prompt1 | llm.with_structured_output(FeedBack)



In [50]:
prompt2 = PromptTemplate(template="Write an appropriate response to this positive feedback: \n {text}",
                         input_variables=["text"])

prompt3 = PromptTemplate(template="write an appropriate response to this negative feedback: \n {text}",
                         input_variables=["text"])

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == "positive", prompt2 | llm | parser),
    (lambda x:x.sentiment == "negative", prompt3 | llm | parser ),
    RunnableLambda(lambda x: "Could not find any Sentiment")
)

chain = classifier_chain | branch_chain

result = chain.invoke({"text" : "My name is aadi and this is a Good product"})
print(result)

Here are a few examples of responses that acknowledge and appreciate the positive feedback:

1. "Thank you so much for your kind words! I'm thrilled to hear that my efforts have made a positive impact."
2. "I really appreciate your feedback - it means a lot to me to know that I've been able to make a difference."
3. "Your support and encouragement mean the world to me. Thank you for believing in me!"
4. "I'm grateful for your recognition of my hard work. It's always great to hear that my efforts are appreciated."
5. "Thank you for taking the time to share your positive feedback with me. It's a wonderful boost to my confidence!"

These responses acknowledge the person's kind words, express gratitude, and show appreciation for their support. Feel free to modify them to fit your personal style and tone!


In [34]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
